# Published-recipe qualification
Use Internet, GPU T4 x2, and the GH_TOKEN secret. Run implementation first, inspect its results, then use fresh timing/development notebook versions. Development uses one common budget selected from both task timings. No final study or confirmation evaluation is started by this notebook.


In [ ]:
# Change these settings for the stage you intend to run.
STAGE = "implementation"  # implementation, timing, development
FAMILY = "mqar"           # mqar or overwrite
BACKEND = "reference"     # choose after inspecting the implementation pilot
EPOCHS = None              # required for development; use the same budget for both tasks
QUALIFICATION_BRANCH = "" # diagnostic/recipe-pilot/... when required
TIMING_BRANCH = ""        # diagnostic/published-recipe/<family>/timing/...

import os, pathlib, subprocess, sys, tempfile, time, json
from kaggle_secrets import UserSecretsClient

started = time.monotonic()
def remaining():
    budget = int(41400 - (time.monotonic() - started) - 30)
    if budget <= 900:
        raise RuntimeError("Notebook time reserve reached")
    return budget

def command(arguments, capture=False):
    if capture:
        return subprocess.check_output(arguments, text=True, timeout=remaining())
    return subprocess.run(arguments, check=True, timeout=remaining())

assert STAGE in ("implementation", "timing", "development")
assert FAMILY in ("mqar", "overwrite")
assert BACKEND in ("reference", "chunked")
repo = pathlib.Path("/kaggle/working/Major-Project-Research")
if not repo.exists():
    command(["git", "clone", "--branch", "maink",
             "https://github.com/code-runner1306/Major-Project-Research.git", str(repo)])
os.chdir(repo)
command(["git", "fetch", "origin", "maink"])
command(["git", "checkout", "--detach", "origin/maink"])
print("Source:", command(["git", "rev-parse", "HEAD"], capture=True).strip())
command([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
print("Setup seconds:", round(time.monotonic() - started, 1))

# Read only the named JSON evidence files, without merging result branches.
def evidence(branch, kind):
    command(["git", "check-ref-format", "refs/heads/" + branch])
    ref = "refs/remotes/origin/" + branch
    command(["git", "fetch", "origin", "refs/heads/" + branch + ":" + ref])
    root = pathlib.Path(tempfile.mkdtemp(prefix="recipe_evidence_", dir="/kaggle/working"))
    names = (["pilot_status.json", "numerical_qualification.json"] if kind == "implementation" else
             ["session_status.json", "scalar/summary.json", "channel-wise/summary.json"])
    for name in names:
        text = command(["git", "show", ref + ":results/" + name], capture=True)
        json.loads(text)
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text)
    return str(root)

os.environ.update(PYTHON_BIN=sys.executable, GIT_PUSH="1",
                  OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
os.environ["SESSION_REMAINING_SECONDS"] = str(remaining())
if STAGE == "implementation":
    launcher = "run_kaggle_recipe_pilot.sh"
else:
    os.environ.update(FAMILY=FAMILY, MODE=STAGE, BACKEND=BACKEND)
    if BACKEND == "chunked":
        assert QUALIFICATION_BRANCH, "Supply the successful GPU implementation branch"
        os.environ["QUALIFICATION_ROOT"] = evidence(QUALIFICATION_BRANCH, "implementation")
    if STAGE == "development":
        assert EPOCHS is not None and TIMING_BRANCH, "Choose the common epoch budget and timing branch first"
        os.environ["EPOCHS"] = str(EPOCHS)
        os.environ["TIMING_EVIDENCE_ROOT"] = evidence(TIMING_BRANCH, "timing")
    left = remaining()
    os.environ["SESSION_REMAINING_SECONDS"] = str(left)
    os.environ["TIME_BUDGET_SECONDS"] = str(min(39600, left - 600))
    launcher = "run_kaggle_published_recipe.sh"
command(["bash", launcher])
print("Total notebook seconds:", round(time.monotonic() - started, 1))
